In [ ]:

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

file_path = "Text_Similarity_Dataset.csv"

df = pd.read_csv(file_path)

print("=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)

print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst 5 Rows:")

print(
    df[
        ["Unique_ID"]
    ].head()
)

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

print(df.isnull().sum())

data = df[
    ["Unique_ID", "text1", "text2"]
].copy()

data = data.dropna(
    subset=["text1", "text2"]
)

data = data.reset_index(drop=True)

print("\nDataset after removing missing values:")
print(data.shape)
def preprocess_text(text):

    # Convert to string
    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        "",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        "",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        "",
        text
    )

    # Remove punctuation
    text = re.sub(
        r"[^a-zA-Z\s]",
        "",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()
data["text1_clean"] = data[
    "text1"
].apply(preprocess_text)

data["text2_clean"] = data[
    "text2"
].apply(preprocess_text)


print("\nText preprocessing completed.")
print("\n" + "=" * 60)
print("TF-IDF VECTORIZATION")
print("=" * 60)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=30000
)
combined_text = pd.concat(
    [
        data["text1_clean"],
        data["text2_clean"]
    ],
    ignore_index=True
)

vectorizer.fit(combined_text)
tfidf_text1 = vectorizer.transform(
    data["text1_clean"]
)

tfidf_text2 = vectorizer.transform(
    data["text2_clean"]
)


print(
    "Text1 TF-IDF Shape:",
    tfidf_text1.shape
)

print(
    "Text2 TF-IDF Shape:",
    tfidf_text2.shape
)

print("\n" + "=" * 60)
print("CALCULATING COSINE SIMILARITY")
print("=" * 60)

similarity_scores = (
    tfidf_text1.multiply(tfidf_text2)
    .sum(axis=1)
)

similarity_scores = np.asarray(
    similarity_scores
).flatten()
data["similarity_percentage"] = (
    similarity_scores * 100
)

data["similarity_percentage"] = (
    data["similarity_percentage"]
    .round(2)
)
def classify_similarity(score):

    if score >= 70:
        return "High Similarity / Potential Plagiarism"

    elif score >= 40:
        return "Moderate Similarity"

    elif score >= 20:
        return "Low Similarity"

    else:
        return "Very Low Similarity"


data["plagiarism_status"] = (
    data["similarity_percentage"]
    .apply(classify_similarity)
)

print("\n" + "=" * 60)
print("SIMILARITY RESULTS")
print("=" * 60)

print(
    data[
        [
            "Unique_ID",
            "similarity_percentage",
            "plagiarism_status"
        ]
    ].head(20).to_string(index=False)
)
print("\n" + "=" * 60)
print("SIMILARITY STATISTICS")
print("=" * 60)

average_similarity = (
    data["similarity_percentage"].mean()
)

maximum_similarity = (
    data["similarity_percentage"].max()
)

minimum_similarity = (
    data["similarity_percentage"].min()
)

print(
    "Average Similarity:",
    round(average_similarity, 2),
    "%"
)

print(
    "Maximum Similarity:",
    round(maximum_similarity, 2),
    "%"
)

print(
    "Minimum Similarity:",
    round(minimum_similarity, 2),
    "%"
)
print("\n" + "=" * 60)
print("SIMILARITY CATEGORY COUNT")
print("=" * 60)

print(
    data["plagiarism_status"]
    .value_counts()
)
print("\n" + "=" * 60)
print("TOP 10 MOST SIMILAR PAIRS")
print("=" * 60)

top_10 = data.sort_values(
    by="similarity_percentage",
    ascending=False
).head(10)

print(
    top_10[
        [
            "Unique_ID",
            "similarity_percentage",
            "plagiarism_status"
        ]
    ].to_string(index=False)
)

high_similarity_count = len(
    data[
        data["similarity_percentage"] >= 70
    ]
)

print("\n" + "=" * 60)
print("PLAGIARISM SUMMARY")
print("=" * 60)

print(
    "Potential plagiarism pairs:",
    high_similarity_count
)
results = data[
    [
        "Unique_ID",
        "text1",
        "text2",
        "similarity_percentage",
        "plagiarism_status"
    ]
].copy()

output_file = (
    "Plagiarism_Detection_Results.csv"
)

results.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 60)
print("RESULTS SAVED")
print("=" * 60)

print(
    "Output file:",
    output_file
)
def check_plagiarism(
    student_text,
    reference_text
):

    # Preprocess
    student_text = preprocess_text(
        student_text
    )

    reference_text = preprocess_text(
        reference_text
    )


    # Create TF-IDF vectorizer
    realtime_vectorizer = TfidfVectorizer(
        stop_words="english"
    )


    # Convert text to vectors
    vectors = realtime_vectorizer.fit_transform(
        [
            student_text,
            reference_text
        ]
    )


    # Cosine similarity
    similarity = cosine_similarity(
        vectors[0:1],
        vectors[1:2]
    )[0][0]


    # Percentage
    percentage = round(
        similarity * 100,
        2
    )


    # Classification
    status = classify_similarity(
        percentage
    )


    return percentage, status

print("\n" + "=" * 60)
print("REAL-TIME PLAGIARISM DETECTOR")
print("=" * 60)


student_assignment = """
Natural language processing enables computers
to understand human language and analyze text.
"""


reference_assignment = """
Natural language processing allows computers
to understand human language and process text.
"""


score, status = check_plagiarism(
    student_assignment,
    reference_assignment
)


print(
    "Test Similarity:",
    score,
    "%"
)

print(
    "Test Status:",
    status
)
print("\n" + "=" * 60)
print("CHECK YOUR OWN ASSIGNMENT")
print("=" * 60)

print(
    "\nEnter student assignment text."
)

user_student_text = input(
    "Student Text: "
)

print(
    "\nEnter reference/source text."
)

user_reference_text = input(
    "Reference Text: "
)
user_score, user_status = check_plagiarism(
    user_student_text,
    user_reference_text
)

print("\n" + "=" * 60)
print("FINAL PLAGIARISM RESULT")
print("=" * 60)

print(
    "Similarity Percentage:",
    user_score,
    "%"
)

print(
    "Status:",
    user_status
)

print("=" * 60)
print("PROGRAM COMPLETED")
print("=" * 60)

DATASET LOADED SUCCESSFULLY
Dataset Shape: (4023, 3)
Columns: ['Unique_ID', 'text1', 'text2']

First 5 Rows:
   Unique_ID
0          0
1          1
2          2
3          3
4          4

MISSING VALUES
Unique_ID    0
text1        0
text2        0
dtype: int64

Dataset after removing missing values:
(4023, 3)

Text preprocessing completed.

TF-IDF VECTORIZATION
Text1 TF-IDF Shape: (4023, 30000)
Text2 TF-IDF Shape: (4023, 30000)

CALCULATING COSINE SIMILARITY

SIMILARITY RESULTS
 Unique_ID  similarity_percentage   plagiarism_status
         0                   1.17 Very Low Similarity
         1                   2.77 Very Low Similarity
         2                   3.56 Very Low Similarity
         3                   2.50 Very Low Similarity
         4                   4.43 Very Low Similarity
         5                   3.27 Very Low Similarity
         6                   1.61 Very Low Similarity
         7                  12.26 Very Low Similarity
         8                   6.